# Fault SIMON64/96: Differential Fault Analysis Assignment

**Target:** the official NSA `simon-speck-supercop` SIMON64/96 reference `Encrypt()` (unmodified) on CW308_STM32F3

**Scope:** CW-Husky / OpenADC

**Goal:** recover the complete 96-bit SIMON64/96 master key using only clock-glitch fault injection and the known plaintext/ciphertext pair -- no subkey is ever given to you as ground truth, and your code must never assume or check against one. You are given:

- the exact `ext_offset` for each of the three targeted subkeys (`key[39]`, `key[40]`, `key[41]`),
- a `width`/`offset` range known to contain a working glitch setting for each,
- the recovery formulas for `key[39]` and `key[40]` (each needs only a later, already-recovered subkey -- never the one it's solving for),
- the key-schedule inversion recurrence.

**What you have to build:** the sweep that actually finds a working `(width, offset)` point within each given range, repeats the injection enough times to separate a real, reproducible fault from noise, and chains the three recovered subkeys into the full master key.

**Validating your own answer without ground truth:** once you have a candidate master key, re-encrypt the known plaintext with it and compare against the *plain* (non-glitched) captured ciphertext -- that comparison needs no secret subkey, just the public plaintext/ciphertext pair every attacker in this scenario already has.

## Firmware and attack background

`simpleserial-simon-fault.c` wraps the OFFICIAL NSA `simon-speck-supercop` reference (`crypto_stream/simon6496ctr/ref/stream.c`), calling `ExpandKey()`/`Encrypt()` from the outside only. `trigger_high()`/`trigger_low()` bracket **only** the `Encrypt()` call itself, so `ext_offset=0` is the very first instruction of `Encrypt()`, and `ext_offset` is measured in ADC samples from there (with `scope.clock.adc_mul = 1`, one sample = one target clock cycle).

SIMON64/96's round structure: `f32(x) = (ROL32(x,1) \& ROL32(x,8)) \^ ROL32(x,2)`, and each of the 42 rounds (indices `0..41`) does:
```
y ^= f32(x); y ^= rk[i]
x ^= f32(y); x ^= rk[i+1]
```
for `i = 0, 2, ..., 40`. The final ciphertext is `(y, x)` after the last round (`i=40`). Note that `rk[41]` is literally the very LAST thing XORed into the output -- that's why `key[41]` is the easiest target.

In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CW308_STM32F3'
SS_VER = 'SS_VER_1_1'   # simpleserial-simon-fault.c uses the default SimpleSerial v1 protocol


## Target firmware

Build `simpleserial-simon-fault` for the target.

In [ ]:
#%%bash -s "$PLATFORM"
## Force the correct ARM toolchain (14.2.1, /usr/bin) ahead of the older
## arm-none-eabi-gcc Xilinx's settings64.sh (sourced in ~/.bashrc) prepends onto PATH.
#export PATH="/usr/bin:$PATH"
#cd ../../../firmware/mcu/simpleserial-simon-fault
#make PLATFORM=$1 -j


## Attack setup

### CW connection and target flashing

In [ ]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"


Flash the target:

In [ ]:
fw_path = "../../../firmware/mcu/simpleserial-simon-fault/simpleserial-simon-fault-{}.hex".format(PLATFORM)
cw.program_target(scope, prog, fw_path)


In [ ]:
def reboot_flush():
    reset_target(scope)
    target.flush()


### Protocol helpers and the official SIMON64/96 test vector

This is the fixed, public NSA test vector (NSA Implementation Guide Sec. 9) -- not a secret, used throughout:

```
pt = 63 6c 69 6e 67 20 72 6f
k  = 00 01 02 03 08 09 0a 0b 10 11 12 13
ct = c8 8f 1a 11 7f e2 a2 5c
```

Your attack must never read `test_k` directly -- it exists only so the harness can flash a known key onto the target and so your FINAL answer can be self-checked by re-encryption, not by peeking.

In [ ]:
import time
import struct
from collections import Counter

def send_key(k_bytes):
    """k_bytes: 12 bytes, K[0..2] little-endian words (as in the test vector)."""
    assert len(k_bytes) == 12
    target.simpleserial_write('k', bytearray(k_bytes))
    return target.simpleserial_read_witherrors('r', 1, glitch_timeout=10)

def send_pt(pt_bytes, glitch_timeout=10, timeout=200):
    """pt_bytes: 8 bytes, (Pt[0],Pt[1]) little-endian words. Returns the simpleserial result dict."""
    assert len(pt_bytes) == 8
    target.simpleserial_write('p', bytearray(pt_bytes))
    return target.simpleserial_read_witherrors('r', 8, glitch_timeout=glitch_timeout, timeout=timeout)

test_pt  = bytes.fromhex('636c696e6720726f')
test_k   = bytes([0x00,0x01,0x02,0x03, 0x08,0x09,0x0a,0x0b, 0x10,0x11,0x12,0x13])
test_ct_expected = bytes([0xc8,0x8f,0x1a,0x11,0x7f,0xe2,0xa2,0x5c])


### First execution

Confirm the firmware reproduces the official test vector before introducing any faults.

In [ ]:
reboot_flush()
send_key(test_k)
val = send_pt(test_pt)
assert val['valid'], val
ct = bytes(val['payload'])
print("ciphertext:", ct.hex())
print("expected:  ", test_ct_expected.hex())
assert ct == test_ct_expected, "Firmware does not reproduce the official SIMON64/96 test vector!"
print("PASS -- firmware matches the official test vector")


### Glitch setup (done once)

- `scope.clock.adc_mul = 1` gives the 1:1 ADC-sample-to-target-cycle ratio that makes `ext_offset` directly usable as a cycle count.
- On Husky, `cglitch_setup()` wires up the glitch circuit but does not turn it on -- `scope.glitch.enabled = True` is required separately.

In [ ]:
scope.cglitch_setup()
scope.clock.adc_mul = 1
if scope._is_husky:
    scope.glitch.enabled = True
scope.adc.samples = 600

reboot_flush()


### Full trace, glitches off

Capture one clean `Encrypt()` call (glitches disabled) and plot it, marking the 21 macro-iterations -- the last three carry `key[39]`/`key[40]`/`key[41]`.

In [ ]:
import holoviews as hv
from holoviews import opts
hv.extension('bokeh')

scope.io.hs2 = "clkgen"   # glitches off for this reference capture

reboot_flush()
send_key(test_k)

scope.arm()
target.simpleserial_write('p', bytearray(test_pt))
ret = scope.capture()
if ret:
    print("Timeout - no trigger")
wave = scope.get_last_trace()
val = target.simpleserial_read_witherrors('r', 8, glitch_timeout=10, timeout=200)
goldciph = bytes(val['payload']) if val['valid'] else None
print("Ciphertext:", goldciph.hex() if goldciph else val)
assert goldciph == test_ct_expected

trigger_samples = scope.adc.trig_count
ITER_LEN = trigger_samples / 21.0

curve = hv.Curve(wave[:trigger_samples]).opts(width=900, height=350)
iter_lines = hv.Overlay([hv.VLine(i * ITER_LEN).opts(color="gray", line_dash="dotted", line_width=1)
                          for i in range(22)])
(curve * iter_lines)


### A single glitch, visualized

One glitch at `ext_offset=372` (`key[41]`'s own position), plotted as a direct pre-campaign sanity check -- used here only to see the effect, not to recover anything yet.

In [ ]:
scope.io.hs2 = "glitch"
scope.glitch.clk_src = "clkgen"
scope.glitch.trigger_src = "ext_single"
scope.glitch.width = 245     # midpoint of the known-good 220-270 range
scope.glitch.offset = 225    # midpoint of the known-good 190-260 range
scope.glitch.ext_offset = 372
scope.glitch.repeat = 1

reboot_flush()
send_key(test_k)

scope.arm()
target.simpleserial_write('p', bytearray(test_pt))
ret = scope.capture()
wave_test = scope.get_last_trace()
val = target.simpleserial_read_witherrors('r', 8, glitch_timeout=10, timeout=200)
print("glitched response:", val)

(hv.Curve(wave_test[:trigger_samples]).opts(width=900, height=350) *
 hv.Path([(scope.glitch.ext_offset, min(wave_test)), (scope.glitch.ext_offset, max(wave_test))]).opts(color="red"))


### Structural classifier (given)

These helpers need no secret key -- only the plaintext/ciphertext pair and SIMON's own public round function:

- `classify()` separates a clean Target-A leak (`dCt0==0`, real signature) from the known Target-C dead end (same `dCt0==0` shape, but `dCt1` equals the publicly-computable `ROL(Ct0,1) & ROL(Ct0,8)` -- not a useful fault) from an ambiguous Target-B-type leak (`dCt0 != 0`, resolved later by its own peel formula) from anything else.
- `is_addr_garbage()` flags STM32F3 flash/SRAM pointers or implausibly small values leaking out over simpleserial from a crashed/diverted return path, rather than real (possibly-faulty) ciphertext.

In [ ]:
ROL32 = lambda x, r: ((x << r) | (x >> (32 - r))) & 0xffffffff
ROR32 = lambda x, r: ((x >> r) | (x << (32 - r))) & 0xffffffff
f32   = lambda x: (ROL32(x, 1) & ROL32(x, 8)) ^ ROL32(x, 2)

def unpack_ct(payload):
    b = bytes(payload)
    Ct0 = struct.unpack_from('<I', b, 0)[0]
    Ct1 = struct.unpack_from('<I', b, 4)[0]
    return Ct0, Ct1

def classify(ct_good, ct_glitched):

    Ct0, Ct1 = ct_good
    Ct0f, Ct1f = ct_glitched
    dCt0 = Ct0 ^ Ct0f
    dCt1 = Ct1 ^ Ct1f

    if dCt0 == 0 and dCt1 != 0:
        predicted_deadend = ROL32(Ct0, 1) & ROL32(Ct0, 8)
        if dCt1 == predicted_deadend:
            return 'A_deadend'
        return 'A_signature'
    elif dCt0 != 0:
        return 'ambiguous'
    else:
        return 'other'

def is_addr_garbage(Ct0f, Ct1f):
    """Flags STM32F3 flash/SRAM pointers (0x0800xxxx / 0x2000xxxx), or small integers that cannot
    plausibly be a real 32-bit ciphertext half."""
    def looks_like_addr(v):
        return (0x08000000 <= v <= 0x080FFFFF) or (0x20000000 <= v <= 0x2001FFFF) or v < 0x1000
    return looks_like_addr(Ct0f) or looks_like_addr(Ct1f)

test_ct_good = unpack_ct(test_ct_expected)


### Given: where to glitch

| subkey | `ext_offset` | `width` range | `offset` range |
|---|---|---|---|
| `key[41]` | 372 | 220-270 | 190-260 |
| `key[39]` | 354 | 220-270 | 190-260 |
| `key[40]` | 374 | 220-270 | 190-260 |

Only a fraction of `(width, offset)` in each range lands cleanly -- the sweep below finds the productive points itself and repeats each enough to separate a real, reproducible fault from noise.

### Campaign engine (given)

One reusable sweep for all three subkeys: fixed `ext_offset`, full box swept at step 1, several trials/point. Each campaign builds its own `chipwhisperer.GlitchController` -- `cw.GlitchController(groups=[...], parameters=["width","offset","ext_offset"])` -- and `run_campaign()` configures its ranges, turns on its built-in live display (`gc.display_stats()`), and sweeps it via `gc.glitch_values()`, logging every capture with `gc.add(group, (w, o, ext_offset))`. Each campaign also supplies its own `candidate_fn(Ct0f, Ct1f)` for the recovery formula/filter logic, returning a candidate or `None` to skip a capture that doesn't qualify for that attack.


In [ ]:
import chipwhisperer as cw
from IPython.display import display

GLITCH_PLOTDOTS = {
    "A_signature": "+g",   # green +
    "A_deadend":   ".k",   # black dot
    "ambiguous":   "^y",   # yellow triangle
    "other":       None,   # not plotted, same as normal
    "reset":       "xr",   # red x
    "normal":      None,   # not plotted
}

def run_campaign(ext_offset, widths, offsets, trials_per_point, candidate_fn, label, gc):
    
    gc.set_range("width", min(widths), max(widths))
    gc.set_range("offset", min(offsets), max(offsets))
    gc.set_range("ext_offset", min(ext_offset), max(ext_offset))
    gc.display_stats()
    display(gc.glitch_plot(plotdots=GLITCH_PLOTDOTS, x_index="width", y_index="offset"))

    scope.io.hs2 = "glitch"
    scope.glitch.clk_src = "clkgen"
    scope.glitch.trigger_src = "ext_single"
    scope.glitch.repeat = 1

    reboot_flush()
    send_key(test_k)
    send_pt(test_pt)

    tally = Counter()
    points = {}
    total = len(widths) * len(offsets) * len(ext_offset)
    n_done = 0
    start_time = time.time()

    for w, o, eo in gc.glitch_values():
        w, o, eo = int(w), int(o), int(eo)
        scope.glitch.width = w
        scope.glitch.offset = o
        scope.glitch.ext_offset = eo
        n_done += 1

        for _ in range(trials_per_point):
            target.flush()
            scope.arm()
            target.simpleserial_write('p', bytearray(test_pt))
            ret = scope.capture()
            if ret:
                gc.add("reset", (w, o, eo))
                reboot_flush(); send_key(test_k); time.sleep(0.02)
                continue

            val = target.simpleserial_read_witherrors('r', 8, glitch_timeout=1, timeout=100)
            if not val['valid'] or val['payload'] is None or len(val['payload']) != 8:
                gc.add("reset", (w, o, eo))
                reboot_flush(); send_key(test_k); time.sleep(0.02)
                continue

            ct_glitched = unpack_ct(val['payload'])
            if ct_glitched == test_ct_good:
                gc.add("normal", (w, o, eo))
                continue

            Ct0f, Ct1f = ct_glitched
            bucket = classify(test_ct_good, ct_glitched)
            gc.add(bucket, (w, o, eo))

            candidate = candidate_fn(Ct0f, Ct1f)
            if candidate is not None:
                tally[candidate] += 1
                points.setdefault(candidate, []).append((w, o, eo))

        if n_done % 500 == 0:
            elapsed = time.time() - start_time
            eta_min = (total - n_done) * (elapsed / n_done) / 60
            print(f"[{label}] {n_done}/{total} points, {elapsed/60:.1f} min elapsed, "
                  f"~{eta_min:.1f} min remaining")

    print(f"\n[{label}] group counts:", dict(zip(gc.groups, gc.group_counts)))
    print(f"[{label}] candidate tally:", tally.most_common(5))
    recovered = tally.most_common(1)[0][0] if tally else None
    if recovered is not None:
        print(f"[{label}] points behind the top candidate:", points[recovered])
    print(f"[{label}] recovered = {recovered:08x}" if recovered is not None else f"[{label}] not yet recovered")
    return recovered, tally, points


### Given: recovery formulas

Each needs only the plaintext/ciphertext and `key[41]` (once recovered) -- never the subkey being solved for, never ground truth.

In [ ]:
def key39_hypothesis(Ct0f, Ct1f, key41):
    """Valid if this sample was actually a Target-A' fault on the 2nd-to-last iteration (i=38).
    Needs only key41 (already recovered) and the known ciphertext(test_ct_good) -- never key39 or key40."""
    Ct0, Ct1 = test_ct_good
    x40        = Ct1  ^ f32(Ct0)  ^ key41
    x40_faulty = Ct1f ^ f32(Ct0f) ^ key41
    return x40 ^ x40_faulty

def key39_mechanism_consistent(Ct0f, Ct1f, key41):
    """A clean skip of key[39]'s XOR only disturbs x entering round 40 -- y_pre and key40 are
    identical between the good and faulty run, so they cancel completely out of Ct0_good^Ct0f."""
    Ct0, Ct1 = test_ct_good
    x40 = Ct1 ^ f32(Ct0) ^ key41
    x40_faulty = Ct1f ^ f32(Ct0f) ^ key41
    return (Ct0 ^ Ct0f) == (f32(x40) ^ f32(x40_faulty))

def key40_hypothesis(Ct0f, Ct1f, key41):
    """Needs only key41 (already recovered) and the known ciphertext(test_ct_good) -- never key40 itself."""
    Ct0, Ct1 = test_ct_good
    x_pre = Ct1f ^ key41 ^ f32(Ct0f)
    return Ct0 ^ Ct0f ^ f32(x_pre)

def invert_key_schedule(key39, key40, key41):
    """Walks the SAME recurrence as the official ExpandKey() (simon_ref/stream.c) backward."""
    rk = [0] * 42
    rk[39], rk[40], rk[41] = key39, key40, key41
    c = 0xfffffffc
    z = 0x7369f885192c0ef5
    for i in range(41, 2, -1):
        zbit = (z >> (i - 3)) & 1
        rk[i - 3] = rk[i] ^ c ^ zbit ^ ROR32(rk[i - 1], 3) ^ ROR32(rk[i - 1], 4)
    return [rk[0], rk[1], rk[2]]


### Campaign 1 -- `key[41]`

In [ ]:
gc41 = cw.GlitchController(groups=["A_signature", "A_deadend", "ambiguous", "other", "reset", "normal"],
                            parameters=["width", "offset", "ext_offset"])

def key41_candidate(Ct0f, Ct1f):
    if classify(test_ct_good, (Ct0f, Ct1f)) == 'A_signature':
        return Ct1f ^ test_ct_good[1]
    return None

key41_recovered, tally41, points41 = run_campaign(
    ext_offset=range(372,373), widths=range(220, 271), offsets=range(190, 261),
    trials_per_point=3, candidate_fn=key41_candidate, label="key41", gc=gc41)


### Campaign 2 -- `key[39]`

In [ ]:
assert key41_recovered is not None, "recover key[41] first"

gc39 = cw.GlitchController(groups=["A_signature", "A_deadend", "ambiguous", "other", "reset", "normal"],
                            parameters=["width", "offset", "ext_offset"])

def key39_candidate(Ct0f, Ct1f):
    if classify(test_ct_good, (Ct0f, Ct1f)) == 'ambiguous' \
            and not is_addr_garbage(Ct0f, Ct1f) \
            and key39_mechanism_consistent(Ct0f, Ct1f, key41_recovered):
        return key39_hypothesis(Ct0f, Ct1f, key41_recovered)
    return None

key39_recovered, tally39, points39 = run_campaign(
    ext_offset=range(354,355), widths=range(220, 271), offsets=range(190, 261),
    trials_per_point=3, candidate_fn=key39_candidate, label="key39", gc=gc39)


### Campaign 3 -- `key[40]`

In [ ]:
assert key41_recovered is not None, "recover key[41] first"

gc40 = cw.GlitchController(groups=["A_signature", "A_deadend", "ambiguous", "other", "reset", "normal"],
                            parameters=["width", "offset", "ext_offset"])

def key40_candidate(Ct0f, Ct1f):
    if classify(test_ct_good, (Ct0f, Ct1f)) == 'ambiguous' and not is_addr_garbage(Ct0f, Ct1f):
        return key40_hypothesis(Ct0f, Ct1f, key41_recovered)
    return None

key40_recovered, tally40, points40 = run_campaign(
    ext_offset=range(374, 375), widths=range(220, 271), offsets=range(190, 261),
    trials_per_point=3, candidate_fn=key40_candidate, label="key40", gc=gc40)


### Reverse propagation and self-check

Invert the schedule, then validate without ground truth: feed the recovered master key back into
`send_key()`/`send_pt()` and let the SAME firmware `Encrypt()` re-encrypt the known plaintext, then
compare the result against `test_ct_expected` -- the real target is the judge, not a Python
re-implementation.


In [ ]:
assert key39_recovered is not None and key40_recovered is not None and key41_recovered is not None, \
    "recover all three subkeys first"

recovered_K = invert_key_schedule(key39_recovered, key40_recovered, key41_recovered)
print(f"recovered master key words: {[f'{w:08x}' for w in recovered_K]}")

recovered_k_bytes = struct.pack('<III', *recovered_K)

scope.io.hs2 = "clkgen"   # glitches off -- this is a plain verification encrypt
reboot_flush()
send_key(recovered_k_bytes)
val = send_pt(test_pt)
assert val['valid'], val
reencrypted_ct = bytes(val['payload'])

print(f"firmware re-encrypt with recovered key: {reencrypted_ct.hex()}")
print(f"plain captured ciphertext:              {test_ct_expected.hex()}")

if reencrypted_ct == test_ct_expected:
    print("\nMASTER KEY RECOVERED -- the SAME firmware Encrypt() reproduces the plain capture "
          "exactly when fed the recovered key")
else:
    print("\nNot there yet -- re-check the three recovered subkeys above.")


## Clean up


In [ ]:
scope.dis()
target.dis()
